<a href="https://colab.research.google.com/github/boeun0905/boeun-study/blob/main/%EC%8A%A4%EB%A7%88%ED%8A%B8%ED%99%88_%EC%A0%9C%EC%96%B4_%EC%8B%9C%EC%8A%A4%ED%85%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U langgraph langchain-openai langchain-community duckduckgo-search
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path="/content/.env", override=True)

print(".env 내 OPENAI_API_KEY가 환경변수에 할당됐습니다:", os.environ["OPENAI_API_KEY"][:5]+"*****")
print(".env 내 TAVILY_API_KEY가 환경변수에 할당됐습니다:", os.environ["TAVILY_API_KEY"][:5]+"*****")

.env 내 OPENAI_API_KEY가 환경변수에 할당됐습니다: sk-pr*****
.env 내 TAVILY_API_KEY가 환경변수에 할당됐습니다: tvly-*****


In [ ]:
from typing import TypedDict, List, Dict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.messages import HumanMessage
import json

# 1. 상태 정의
class SmartHomeState(TypedDict):
    city: str
    weather_info: str
    device_status: Dict[str, str]  # {'heating': 'on/off', 'ac': 'on/off', 'cleaner': 'on/off'}

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
search = DuckDuckGoSearchRun()

# 2. 노드 정의

def search_weather_node(state: SmartHomeState):
    """DuckDuckGo를 사용하여 특정 도시의 현재 날씨 검색"""
    city = state["city"]
    query = f"현재 {city} 의 날씨를 알려줘"
    search_result = search.run(query)
    #print(search_result)
    return {"weather_info": search_result}

def device_control_node(state: SmartHomeState):
    """날씨 정보를 바탕으로 기기 제어 로직 결정 """
    weather_info = state["weather_info"]

    prompt = f"""
   아래 날씨 정보를 바탕으로 집안 기기들의 작동 상태를 결정하세요.
    규칙:
    1. 난방: 기온이 낮으면(15°C 미만) '켜짐', 그렇지 않으면 '꺼짐'
    2. 에어컨: 기온이 높으면(25°C 이상) '켜짐', 그렇지 않으면 '꺼짐'
    3. 로봇 청소기: 비나 눈이 오지 않으면 '켜짐', 그렇지 않으면 '꺼짐'
    날씨 정보: {weather_info}
    응답은 다음과 같은 JSON 형식으로만 작성해 주세요.
    {{"heating": "on", "ac": "off", "cleaner": "on"}}
    """

    response = llm.invoke([HumanMessage(content=prompt)])

    try:
        status = json.loads(response.content.strip())
    except:
        status = {"heating": "off", "ac": "off", "cleaner": "off"}

    return {"device_status": status}

# 3. 그래프 구성
workflow = StateGraph(SmartHomeState)

# 노드 추가
workflow.add_node("search_weather", search_weather_node)
workflow.add_node("control_devices", device_control_node)

# 연결
workflow.set_entry_point("search_weather")
workflow.add_edge("search_weather", "control_devices")
workflow.add_edge("control_devices", END)

# 컴파일
app = workflow.compile()


In [ ]:
def run_smart_home(city_name: str):
    print(f"--- '{city_name}' 스마트홈 시스템 가동 ---")
    inputs = {"city": city_name}
    result = app.invoke(inputs)
    print("run_smart_home_함수 출력 : ",result)
    print(f"검색된 날씨 요약: {result['weather_info'][:300]}...")
    print("기기 제어 결과:")
    for device, power in result["device_status"].items():
        print(f" - {device.upper()}: {power.upper()}")
    print("-" * 30)

if __name__ == "__main__":

    user_input = input("날씨를 확인할 도시 이름을 입력하세요 (예: Seoul, Tokyo, New York): ").strip()

    if user_input:
        run_smart_home(user_input)
    else:
        print("도시 이름을 입력하지 않았습니다. 프로그램을 종료합니다.")

날씨를 확인할 도시 이름을 입력하세요 (예: Seoul, Tokyo, New York): seoul
--- 'seoul' 스마트홈 시스템 가동 ---
run_smart_home_함수 출력 :  {'city': 'seoul', 'weather_info': 'MSN 날씨 과 (와) 함께 Seoul , 서울 의 오늘, 오늘 밤, 내일에 대한 정확한 시간별 예보와 함께 10일간의 일일 예보 및 기상 레이더를 확인하세요. 6 days ago · Seoul | Hourly weather forecast [날씨] 내일 (24일) 아침까지 곳곳에 눈…주말 추위 계속 / KBS 오늘, 내일, 이번 주 서울 날씨 알아보기. 서울 일기예보. 최저/최고기온, 강수확률, 미세먼지 현황. 이번 주 서울 일기예보까지 원클릭 조회. 완벽한 서울 여행을 위해 여행 전 서울 일기예보를 확인하세요. 서울관광재단 Visit Seoul 에서 확인하는 실시간 날씨 정보. Hourly weather forecast in 서울특별시, 서울시, 대한민국. Check current conditions in 서울특별시, 서울시, 대한민국 with radar, hourly, and more. 또한, 해당 일자의 해와 달의 일출 및 일몰 시간도 함께 안내됩니다. 서울특별시 의 시간대별 상세 날씨 예보에는 다음과 같은 기상 요소들이 요약되어 있습니다: 날씨 상태 아이콘, 실제 기온 및 체감 기온, 강수 확률과 강수량, 습도, 기압, 최대 풍속 및 풍향.', 'device_status': {'heating': 'on', 'ac': 'off', 'cleaner': 'off'}}
검색된 날씨 요약: MSN 날씨 과 (와) 함께 Seoul , 서울 의 오늘, 오늘 밤, 내일에 대한 정확한 시간별 예보와 함께 10일간의 일일 예보 및 기상 레이더를 확인하세요. 6 days ago · Seoul | Hourly weather forecast [날씨] 내일 (24일) 아침까지 곳곳에 눈…주말 추위 계속 / KBS 오늘, 내일